In [2]:
import pandas as pd
import numpy as np

In [3]:
movies = pd.read_csv("../../data/tmdb_movies_data.csv")

In [4]:
movies.head(2)

,id,imdb_id,popularity,budget,revenue,original_title,cast,homepage,director,tagline,...,overview,runtime,genres,production_companies,release_date,vote_count,vote_average,release_year,budget_adj,revenue_adj
0,135397,tt0369610,32.985763,150000000,1513528810,Jurassic World,Chris Pratt|Bryce Dallas Howard|Irrfan Khan|Vi...,http://www.jurassicworld.com/,Colin Trevorrow,The park is open.,...,Twenty-two years after the events of Jurassic ...,124,Action|Adventure|Science Fiction|Thriller,Universal Studios|Amblin Entertainment|Legenda...,6/9/2015,5562,6.5,2015,137999939.3,1.392446e+09
1,76341,tt1392190,28.419936,150000000,378436354,Mad Max: Fury Road,Tom Hardy|Charlize Theron|Hugh Keays-Byrne|Nic...,http://www.madmaxmovie.com/,George Miller,What a Lovely Day.,...,An apocalyptic story set in the furthest reach...,120,Action|Adventure|Science Fiction|Thriller,Village Roadshow Pictures|Kennedy Miller Produ...,5/13/2015,6185,7.1,2015,137999939.3,3.481613e+08


In [5]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10866 entries, 0 to 10865
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    10866 non-null  int64  
 1   imdb_id               10856 non-null  object 
 2   popularity            10866 non-null  float64
 3   budget                10866 non-null  int64  
 4   revenue               10866 non-null  int64  
 5   original_title        10866 non-null  object 
 6   cast                  10790 non-null  object 
 7   homepage              2936 non-null   object 
 8   director              10822 non-null  object 
 9   tagline               8042 non-null   object 
 10  keywords              9373 non-null   object 
 11  overview              10862 non-null  object 
 12  runtime               10866 non-null  int64  
 13  genres                10843 non-null  object 
 14  production_companies  9836 non-null   object 
 15  release_date       

In [6]:
movies.rename(columns={"original_title": "title"}, inplace=True)
movies["title"].head(2)

0        Jurassic World
1    Mad Max: Fury Road
Name: title, dtype: object

The columns we will need from this daatset for movies recommendation are:

* id: will be used later to fetch movies posters from TMDB API.
* title
* genres
* keywords
* overview
* cast
* director
* production_companies

These attributes give the best description for the movies.

In [7]:
movies = movies[["id","title", "genres","keywords", "overview", "cast","director","production_companies"]]

In [8]:
movies.head(2)

,id,title,genres,keywords,overview,cast,director,production_companies
0,135397,Jurassic World,Action|Adventure|Science Fiction|Thriller,monster|dna|tyrannosaurus rex|velociraptor|island,Twenty-two years after the events of Jurassic ...,Chris Pratt|Bryce Dallas Howard|Irrfan Khan|Vi...,Colin Trevorrow,Universal Studios|Amblin Entertainment|Legenda...
1,76341,Mad Max: Fury Road,Action|Adventure|Science Fiction|Thriller,future|chase|post-apocalyptic|dystopia|australia,An apocalyptic story set in the furthest reach...,Tom Hardy|Charlize Theron|Hugh Keays-Byrne|Nic...,George Miller,Village Roadshow Pictures|Kennedy Miller Produ...


In [9]:
movies.isnull().sum()

id                         0
title                      0
genres                    23
keywords                1493
overview                   4
cast                      76
director                  44
production_companies    1030
dtype: int64

In [10]:
movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)
movies.isnull().sum()

id                      0
title                   0
genres                  0
keywords                0
overview                0
cast                    0
director                0
production_companies    0
dtype: int64

In [11]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8667 entries, 0 to 8666
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    8667 non-null   int64 
 1   title                 8667 non-null   object
 2   genres                8667 non-null   object
 3   keywords              8667 non-null   object
 4   overview              8667 non-null   object
 5   cast                  8667 non-null   object
 6   director              8667 non-null   object
 7   production_companies  8667 non-null   object
dtypes: int64(1), object(7)
memory usage: 541.8+ KB


In [12]:
def collapse_names(names):
    """
    This function takes a string of names and returns a string without spaces between first and last name.
    """
    names = names.replace(" ","")
    return names

In [13]:
movies['cast'] = movies['cast'].apply(collapse_names)   
movies['director'] = movies['director'].apply(collapse_names)
movies['production_companies'] = movies['production_companies'].apply(collapse_names)

In [14]:
movies.head(3)

,id,title,genres,keywords,overview,cast,director,production_companies
0,135397,Jurassic World,Action|Adventure|Science Fiction|Thriller,monster|dna|tyrannosaurus rex|velociraptor|island,Twenty-two years after the events of Jurassic ...,ChrisPratt|BryceDallasHoward|IrrfanKhan|Vincen...,ColinTrevorrow,UniversalStudios|AmblinEntertainment|Legendary...
1,76341,Mad Max: Fury Road,Action|Adventure|Science Fiction|Thriller,future|chase|post-apocalyptic|dystopia|australia,An apocalyptic story set in the furthest reach...,TomHardy|CharlizeTheron|HughKeays-Byrne|Nichol...,GeorgeMiller,VillageRoadshowPictures|KennedyMillerProductions
2,262500,Insurgent,Adventure|Science Fiction|Thriller,based on novel|revolution|dystopia|sequel|dyst...,Beatrice Prior must confront her inner demons ...,ShaileneWoodley|TheoJames|KateWinslet|AnselElg...,RobertSchwentke,SummitEntertainment|MandevilleFilms|RedWagonEn...


In [15]:
def convert_to_list(text):
    """
    This function takes a string of words and returns a list.
    """
    text = text.split("|")   
    return text

In [16]:
movies['genres'] = movies['genres'].apply(convert_to_list)
movies['keywords'] = movies['keywords'].apply(convert_to_list)
movies['cast'] = movies['cast'].apply(convert_to_list)
movies['director'] = movies['director'].apply(convert_to_list)
movies['production_companies'] = movies['production_companies'].apply(convert_to_list)

In [17]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [18]:
movies.head(2)

,id,title,genres,keywords,overview,cast,director,production_companies
0,135397,Jurassic World,"[Action, Adventure, Science Fiction, Thriller]","[monster, dna, tyrannosaurus rex, velociraptor...","[Twenty-two, years, after, the, events, of, Ju...","[ChrisPratt, BryceDallasHoward, IrrfanKhan, Vi...",[ColinTrevorrow],"[UniversalStudios, AmblinEntertainment, Legend..."
1,76341,Mad Max: Fury Road,"[Action, Adventure, Science Fiction, Thriller]","[future, chase, post-apocalyptic, dystopia, au...","[An, apocalyptic, story, set, in, the, furthes...","[TomHardy, CharlizeTheron, HughKeays-Byrne, Ni...",[GeorgeMiller],"[VillageRoadshowPictures, KennedyMillerProduct..."


In [19]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['director'] + movies['production_companies']
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))
movies['tags'][0]

"Twenty-two years after the events of Jurassic Park, Isla Nublar now features a fully functioning dinosaur theme park, Jurassic World, as originally envisioned by John Hammond. Action Adventure Science Fiction Thriller monster dna tyrannosaurus rex velociraptor island ChrisPratt BryceDallasHoward IrrfanKhan VincentD'Onofrio NickRobinson ColinTrevorrow UniversalStudios AmblinEntertainment LegendaryPictures FujiTelevisionNetwork Dentsu"

In [20]:
new_movies = movies[['id', 'title', 'tags']]
new_movies.sample(5)

,id,title,tags
8279,571,The Birds,Chic socialite Melanie Daniels enjoys a passin...
2689,72571,Paranormal Activity 3,"In 1988, evil begins to terrorize young sister..."
5559,40969,Terror Train,A masked killer targets six college kids respo...
8062,4780,Obsession,New Orleans businessman Michael Courtlandâ€™s ...
2234,8204,The Spiderwick Chronicles,Upon moving into the run-down Spiderwick Estat...


In [21]:
new_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8667 entries, 0 to 8666
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      8667 non-null   int64 
 1   title   8667 non-null   object
 2   tags    8667 non-null   object
dtypes: int64(1), object(2)
memory usage: 203.3+ KB


In [22]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(analyzer="word" ,max_features= 10000, stop_words='english', max_df=0.7)

In [23]:
vector = cv.fit_transform(new_movies['tags']).toarray()
vector.shape

(8667, 10000)

In [24]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vector)

In [25]:
similarity.shape

(8667, 8667)

In [26]:
def recommend(movie):
    index = new_movies[new_movies['title'] == movie].index[0]
    distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
    for i in distances[1:6]:
        print(new_movies.iloc[i[0]].title)

In [33]:
recommend("San Andreas")

Earthquake
10.5: Apocalypse
Hot Shots! Part Deux
The Towering Inferno
Gray Lady Down


In [31]:
import pickle
pickle.dump(new_movies,open('../model/movie_list.pkl','wb'))
pickle.dump(similarity,open('../model/similarity.pkl','wb'))

In [32]:
new_movies.to_csv("../../data/movies.csv", index=False)